# PCA, geometric intuition: variance and rotating axes

The same made-up flat data as the figures in the Note.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import PCA

## 1. Mean vs variance

Two small datasets with the same mean but different spread.

In [ ]:
a = np.array([-5, 0, 5])
b = np.array([-10, 0, 10])
print("means:", a.mean(), b.mean())        # both 0
# variance: average squared distance from the mean
print("variances:", a.var(), b.var())      # 16.7 and 66.7
print("ratio:", b.var() / a.var())          # 4: twice the spread, four times the variance
print("standard deviations:", a.std(), b.std())

## 2. The flat data

In [ ]:
rng = np.random.default_rng(7)
N = 30
rooms = rng.uniform(1, 5, N)
shops = 3 + rng.normal(0, 0.35, N)                     # grocery shops nearby
washrooms = 0.8 * rooms + rng.normal(0, 0.35, N) + 0.4
washrooms = washrooms * rooms.std() / washrooms.std()  # same spread as rooms
washrooms += 3 - washrooms.mean()
flats = pd.DataFrame({"rooms": rooms, "shops": shops, "washrooms": washrooms})
# variance of each column (ddof=0: divide by n, as in the Note)
flats.var(ddof=0).round(2)

Rooms vs shops: rooms has far more variance, so feature selection keeps rooms.
Rooms vs washrooms: the same variance, so feature selection cannot choose.

## 3. Turning the line: variance of the shadows at every angle

In [ ]:
P = flats[["rooms", "washrooms"]].to_numpy()
centred = P - P.mean(axis=0)

def shadow_variance(deg):
    # unit vector pointing at this angle
    d = np.array([np.cos(np.radians(deg)), np.sin(np.radians(deg))])
    # each point's position along the line, then its variance
    return (centred @ d).var()

angles = np.arange(0, 180, 1)
v = [shadow_variance(a) for a in angles]
best = angles[np.argmax(v)]
print("best angle:", best, " variance:", round(max(v), 2), " at 90 degrees more:", round(shadow_variance(best + 90), 2))

In [ ]:
go.Figure(go.Scatter(x=angles, y=v, mode="lines")).update_layout(
    title="Variance of the shadows as the line turns", xaxis_title="angle (degrees)",
    yaxis_title="variance", template="simple_white")

## 4. Interactive: drag the slider to turn the line

In [ ]:
def frame(deg):
    d = np.array([np.cos(np.radians(deg)), np.sin(np.radians(deg))])
    c = P.mean(axis=0)
    shadow = c + np.outer(centred @ d, d)
    line = np.array([c - 3 * d, c + 3 * d])
    return [go.Scatter(x=P[:, 0], y=P[:, 1], mode="markers", name="flats"),
            go.Scatter(x=line[:, 0], y=line[:, 1], mode="lines", name="line"),
            go.Scatter(x=shadow[:, 0], y=shadow[:, 1], mode="markers", name="shadows")]

steps = list(range(0, 180, 5))
fig = go.Figure(frame(0), frames=[go.Frame(data=frame(a), name=str(a)) for a in steps])
fig.update_layout(template="simple_white", width=650, height=600,
                  xaxis=dict(range=[0, 6], title="rooms"), yaxis=dict(range=[0, 6], title="washrooms", scaleanchor="x"),
                  sliders=[{"steps": [{"label": f"{a}° var {shadow_variance(a):.2f}", "method": "animate",
                                       "args": [[str(a)], {"mode": "immediate", "frame": {"duration": 0}}]}
                                      for a in steps]}])
fig

## 5. The same with scikit-learn's PCA

In [ ]:
pca = PCA(n_components=2)
pca.fit(P)
print("PC1 direction:", pca.components_[0].round(3))   # about 45 degrees: [0.707, 0.707] or its negative
# share of the total variance held by each principal component
print("variance share:", pca.explained_variance_ratio_.round(3))

In [ ]:
# keep only PC1: two columns become one
one_column = PCA(n_components=1).fit_transform(P)
one_column.shape